In [1]:
import os
import sys
import torch
import cv2

# 将父目录加入 path 以便导入 drone_dynamics 和 drone_renderer
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from drone_dynamics import simulate_position_step, solve_attitude_from_thrust_and_goal_vec,update_dg
from drone_renderer import DroneRenderer

# 检查 CUDA
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [ ]:
B = 4  # 批量大小
dt = 0.02  # 时间步长
num_steps = 200  # 模拟步数

mesh_path = "../data/sample/sample.obj"  # 无人机模型路径
renderer = DroneRenderer(
    mesh_path=mesh_path,
    device=device,
    image_size=(480, 640),
    focal_length=500.0
)

# 初始化无人机状态
p = torch.rand(B, 3, device=device) * 2.0  # 位置
v = torch.zeros(B, 3, device=device)  # 速度
a = torch.zeros(B, 3, device=device)  # 加速度
act = torch.zeros(B, 3, device=device)  # 动力分配 (实际物理推力状态)
R = torch.eye(3, device=device).unsqueeze(0).repeat(B, 1, 1)  # 姿态矩阵

# 随机生成一个恒定的期望指令用于测试
act_pred_target = torch.rand(B, 3, device=device) * 15.0 
dg = torch.randn((B, 3), device=device) * 0.2 # 初始化扰动
    
target_pos = torch.rand(B, 3, device=device) * 2.0  # 目标位置
g_vec = torch.tensor([0,0,-9.80665], device=device).unsqueeze(0).repeat(B,1)  # 重力向量向下

act_queue = [torch.zeros(B, 3, device=device) for _ in range(2)] # 动作命令队列，用于模拟控制延迟

for step in range(num_steps):

    R_camera ,T_camera = renderer.compute_view_matrix(p_ros=p, R_ros=R, camera_pitch_deg=10.0)
    rgb_images, depth_images = renderer.render(R=R_camera, T=T_camera,return_tensor=True)
    
    if step % 10 == 0:
        print(f"Step {step}: Position {p[0].detach().cpu().numpy()}")
    
    cv2.imshow('RGB', rgb_images[0].cpu().numpy()[:,:,::-1])
    cv2.imshow('Depth', (depth_images[0]/torch.max(depth_images[0])).cpu().numpy())
    if cv2.waitKey(10) & 0xFF == 27: # ESC exit
        break

    p_old = p.clone()

    dg = update_dg(dg_curr=dg, dt=dt, noise_std=0.04) # 更新扰动
    
    act_queue.append(act_pred_target) 
    current_act_cmd = act_queue.pop(0)

    p, v, a, act = simulate_position_step(
        p=p,
        v=v,
        a=a,
        R=R,
        act=act,
        act_pred=current_act_cmd, 
        dt=dt,
        enable_airmode=True,
        dg=dg,
        v_wind=torch.randn((B,3),device=device)*0.1,
        grad_decay=0.8
    )
    thrust_without_gravity = act - g_vec 
    
    R = solve_attitude_from_thrust_and_goal_vec(
        thrust_vector=thrust_without_gravity, 
        velocity=target_pos - p_old, 
        R_old=R,
        yaw_inertia=5.0, 
        dt=dt,
        yaw_ctl_delay=4.0,  # 跟着参考项目调的参数，后面可能会变
    )

    
    
cv2.destroyAllWindows()




In [ ]:
# 整理





In [ ]:
from drone_env import DroneSimulator

B = 4  # 批量大小
dt = 0.02  # 时间步长
num_steps = 200  # 模拟步数

# 初始化仿真环境
# mesh_path: 从 ipynb 目录看，数据在 ../data
env = DroneSimulator(
    batch_size=B, 
    dt=dt, 
    mesh_path="../data/sample/sample.obj",
    device=device 
)

# 随机生成一个恒定的期望指令用于测试
act_pred_target = torch.rand(B, 3, device=device) * 15.0 
target_pos = torch.rand(B, 3, device=device) * 2.0  # 目标位置

print("Start Simulation Loop...")

for step in range(num_steps):
    # 1. 渲染 (Render)
    rgb_images, depth_images = env.render(camera_pitch=10.0)
    
    if step % 10 == 0:
        print(f"Step {step}: Position {env.p[0].detach().cpu().numpy()}")
    
    # 2. 可视化 (Visualization)
    # 注意：cv2.imshow 在某些远程/headless 环境下可能无法显示窗口
    try:
        cv2.imshow('RGB', rgb_images[0].cpu().numpy()[:,:,::-1])
        cv2.imshow('Depth', (depth_images[0]/torch.max(depth_images[0])).cpu().numpy())
        if cv2.waitKey(10) & 0xFF == 27: # ESC exit
            break
    except Exception as e:
        pass # Ignore display errors in headless

    # 3. 准备控制输入 (Control Input)
    # 原逻辑: velocity = target_pos - p_old
    # 在 step() 被调用前，env.p 即为 p_old
    target_vector = target_pos - env.p

    # 4. 模拟步进 (Simulation Step)
    state = env.step(action_cmd=act_pred_target, target_pos_vector=target_vector)

cv2.destroyAllWindows()

Loading mesh from: ../data/sample/sample.obj
Start Simulation Loop...
Step 0: Position [0.9922992  0.12939513 0.34335312]
Step 10: Position [0.99347657 0.16687597 0.39076844]
Step 20: Position [1.0157597  0.42045832 0.5997583 ]
Step 30: Position [1.0653595  0.92780435 0.9750125 ]
Step 40: Position [1.1407086 1.6759238 1.5076915]
Step 50: Position [1.2400085 2.6482577 2.1858714]
Step 60: Position [1.3623179 3.8280334 2.9979587]
Step 70: Position [1.5063753 5.20199   3.9349515]
Step 80: Position [1.6702691 6.756087  4.9877253]
Step 90: Position [1.8532647 8.477028  6.147505 ]
Step 100: Position [ 2.0541923 10.352687   7.4063883]
Step 110: Position [ 2.2719355 12.371477   8.75731  ]
Step 120: Position [ 2.5050437 14.52327   10.192721 ]
Step 130: Position [ 2.752237 16.797943 11.707713]
Step 140: Position [ 3.0131924 19.187414  13.2966385]
Step 150: Position [ 3.2871518 21.6837    14.953714 ]
Step 160: Position [ 3.5737016 24.279064  16.673557 ]
Step 170: Position [ 3.8711905 26.965685  18